In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve,
    f1_score, average_precision_score, precision_recall_curve
)

In [ ]:
train_df = pd.read_csv('../data/processed/processed_train.csv')
test_df  = pd.read_csv('../data/processed/processed_test.csv')

print('Train shape:', train_df.shape)
print('Test shape: ', test_df.shape)
train_df.head()

In [ ]:
TARGET = 'satisfaction'

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_test  = test_df.drop(columns=[TARGET])
y_test  = test_df[TARGET]

print('Features:', X_train.columns.tolist())

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist,
    n_iter=10,         # 10 random combos
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)
random_search.fit(X_train, y_train)

print('Best params:', random_search.best_params_)
print(f'Best CV F1: {random_search.best_score_:.4f}')
best_params = random_search.best_params_

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    RandomForestClassifier(n_estimators=200, random_state=42),
    X_train, y_train,
    cv=cv,
    scoring=['accuracy', 'roc_auc', 'f1', 'average_precision'],
    return_train_score=True
)

print(f"CV Accuracy:  {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")
print(f"CV ROC-AUC:   {cv_results['test_roc_auc'].mean():.4f} ± {cv_results['test_roc_auc'].std():.4f}")
print(f"CV F1 Score:  {cv_results['test_f1'].mean():.4f} ± {cv_results['test_f1'].std():.4f}")
print(f"CV PR-AUC:    {cv_results['test_average_precision'].mean():.4f} ± {cv_results['test_average_precision'].std():.4f}")